# nb09 -- Architecture comparison + targeted validation
Four candidate architectures, trained once each and compared on both predictive accuracy and interpretability:

1. **fc1_only** -- hidden-only MLP, ReLU, L1 on `fc1` only (nb06 style).
2. **fc1_fc2** -- hidden-only MLP, ReLU, L1 on `fc1` and `fc2` (nb07 style).
3. **fc1_group_fc2** -- hidden-only MLP, ReLU, **group-sparsity** on `fc1` (each gene's full weight row across all 64 hidden units penalized as one group, so genes switch fully on/off) + L1 on `fc2`. Not tried yet until now.
4. **skip_connection** -- direct linear term + hidden term, LeakyReLU (nb08/nb09 style).

All four use the same corrected data pipeline and train/val/test split, so accuracy and interpretability numbers are directly comparable.

Full bootstrap + permutation-null validation (expensive -- 11 extra training runs each) is then run only on the two standout variants, picked from the comparison table rather than validating all four. This run also adds a **rank-1-specific stability check**: does each protein's single top predictor gene replicate across bootstraps, separate from the noisier aggregate top-20 metric used in the previous version of this notebook.

## Environment setup

In [1]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q scanpy anndata scikit-misc
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/covid_citeseq_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab | BASE_PATH = /content/drive/MyDrive/covid_citeseq_project


## GPU check

In [2]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device (training only):', DEVICE)

Device (training only): cuda


## Imports and config

In [3]:
!pip install -q -U "typing_extensions>=4.13"

In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

CHECKPOINT_PATH    = BASE_PATH / 'data' / 'processed' / 'covid_subsampled.h5ad'
GENE_MAPPING_PATH  = BASE_PATH / 'results' / 'tables' / 'nb02_covid_adt_gene_mapping.csv'
RESULTS_DIR        = BASE_PATH / 'results' / 'sparse_mlp_comparison'
MODELS_DIR         = RESULTS_DIR / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_HVGS       = 2000
DROP_GENES   = ['PDPN', 'KDR']  # zero-variance in GEX (see nb02 Step 3b)

HIDDEN_DIM          = 64
L1_LAMBDA_FC1        = 1e-4
L1_LAMBDA_FC2        = 1e-4
L1_LAMBDA_FC1_GROUP  = 1e-3   # group lasso scale differs from plain L1 -- may need tuning, check sparsity output
L1_LAMBDA_DIRECT     = 1e-4

BATCH_SIZE   = 256
NUM_EPOCHS   = 100
PATIENCE     = 15

TEST_SIZE     = 0.15
VAL_SIZE      = 0.15
RANDOM_STATE  = 42

SPARSITY_THRESH = 1e-3   # near-zero cutoff for effective-gene / active-unit counts

# Deep validation budgets -- each variant validated costs 1 + N_BOOTSTRAPS + N_NULL_PERMS
# full training runs. Two variants validated by default; reduce these if runtime is tight.
N_BOOTSTRAPS      = 5
N_NULL_PERMS      = 5
AUX_NUM_EPOCHS    = 40
AUX_PATIENCE      = 8
TOP_K_GENES       = 20

## Load checkpoint and gene mapping

In [5]:
covid = sc.read_h5ad(CHECKPOINT_PATH)

gex_mask = covid.var['feature_types'] == 'Gene Expression'
adt_mask = covid.var['feature_types'] == 'Antibody Capture'
covid_gex = covid[:, gex_mask].copy()
covid_adt = covid[:, adt_mask].copy()

gene_map = pd.read_csv(GENE_MAPPING_PATH)
gene_map = gene_map[~gene_map['gene'].isin(DROP_GENES)]
matched_genes     = gene_map['gene'].tolist()
matched_adt_names = gene_map['adt_name'].tolist()

print(f'GEX: {covid_gex.shape}, ADT: {covid_adt.shape}')
print(f'Matched genes/proteins: {len(matched_genes)}')

GEX: (69090, 24737), ADT: (69090, 192)
Matched genes/proteins: 163


## Gene union (matched + HVG)

In [6]:
covid_gex.layers['counts'] = covid_gex.layers['raw'].copy()

sc.pp.highly_variable_genes(
    covid_gex, n_top_genes=N_HVGS, flavor='seurat_v3', layer='counts',
)
hvg_genes = covid_gex.var_names[covid_gex.var['highly_variable']].tolist()


def build_gene_union(matched_genes: list[str], hvg_genes: list[str]) -> list[str]:
    """Union of matched coupling genes and top HVGs, matched genes always included."""
    return sorted(set(matched_genes) | set(hvg_genes))


gene_union = build_gene_union(matched_genes, hvg_genes)
print(f'Gene union: {len(gene_union)}')

Gene union: 2092


## Normalize and correct (same pipeline as nb06/nb07/nb08)

In [7]:
def normalize_rna(adata_gex: sc.AnnData, gene_union: list[str]) -> sc.AnnData:
    """Log1p(CP10k) normalization on raw counts, restricted to gene_union."""
    adata = adata_gex[:, gene_union].copy()
    adata.X = adata.layers['raw'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata


def clr_normalize(counts: np.ndarray) -> np.ndarray:
    """Centered log-ratio normalization per cell -- standard for CITE-seq ADT counts."""
    log_counts = np.log1p(counts)
    geometric_mean = log_counts.mean(axis=1, keepdims=True)
    return log_counts - geometric_mean


def regress_out_library_size(X: np.ndarray, raw_counts_layer: np.ndarray) -> np.ndarray:
    """Residualize each column of X against log1p(per-cell total raw counts)."""
    lib_size = np.log1p(np.asarray(raw_counts_layer).sum(axis=1)).reshape(-1, 1)
    design = np.column_stack([np.ones(X.shape[0]), lib_size])
    beta, _, _, _ = np.linalg.lstsq(design, X, rcond=None)
    return (X - design @ beta).astype(np.float32)


rna_adata = normalize_rna(covid_gex, gene_union)
X_rna = np.asarray(rna_adata.X.todense()) if hasattr(rna_adata.X, 'todense') else np.asarray(rna_adata.X)
X_rna = X_rna.astype(np.float32)

adt_counts = covid_adt[:, matched_adt_names].layers['raw']
adt_counts = np.asarray(adt_counts.todense()) if hasattr(adt_counts, 'todense') else np.asarray(adt_counts)
Y_protein = clr_normalize(adt_counts).astype(np.float32)

rna_raw_for_size = rna_adata.layers['raw']
rna_raw_for_size = np.asarray(rna_raw_for_size.todense()) if hasattr(rna_raw_for_size, 'todense') else np.asarray(rna_raw_for_size)

X_rna_corr     = regress_out_library_size(X_rna, rna_raw_for_size)
Y_protein_corr = regress_out_library_size(Y_protein, adt_counts)

print(f'X_rna_corr: {X_rna_corr.shape} | Y_protein_corr: {Y_protein_corr.shape}')

X_rna_corr: (69090, 2092) | Y_protein_corr: (69090, 163)


## Train / val / test split

In [8]:
n_cells = X_rna_corr.shape[0]
all_idx = np.arange(n_cells)

train_idx, test_idx = train_test_split(all_idx, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, val_idx  = train_test_split(train_idx, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=RANDOM_STATE)

print(f'Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')

Train: 48,362 | Val: 10,364 | Test: 10,364


## DataLoaders

In [9]:
def make_loader(X: np.ndarray, Y: np.ndarray, idx: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(X[idx]), torch.from_numpy(Y[idx]))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)


train_loader = make_loader(X_rna_corr, Y_protein_corr, train_idx, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_rna_corr, Y_protein_corr, val_idx,   BATCH_SIZE, shuffle=False)

## Architectures
`HiddenOnlyMLP` (ReLU) covers variants 1-3, which differ only in their L1 penalty function. `SkipConnectionSparseMLP` (LeakyReLU) is variant 4.

In [10]:
class HiddenOnlyMLP(nn.Module):
    """RNA -> hidden -> protein, ReLU. Used for the fc1_only / fc1_fc2 / fc1_group_fc2 variants."""

    def __init__(self, rna_dim: int, hidden_dim: int, protein_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(rna_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, protein_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.relu(self.fc1(x)))


class SkipConnectionSparseMLP(nn.Module):
    """RNA -> protein via a direct linear term plus a hidden nonlinear (ReLU) term."""

    def __init__(self, rna_dim: int, hidden_dim: int, protein_dim: int):
        super().__init__()
        self.direct = nn.Linear(rna_dim, protein_dim)
        self.fc1 = nn.Linear(rna_dim, hidden_dim)
        # self.activation = nn.LeakyReLU(negative_slope=0.01)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, protein_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        direct_pred, hidden_pred = self.forward_components(x)
        return direct_pred + hidden_pred

    def forward_components(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        return self.direct(x), self.fc2(self.activation(self.fc1(x)))

## Penalty functions -- one per variant
Group lasso on `fc1`: L2 norm of each gene's full weight column (across all hidden units), summed. Unlike plain L1, this drives whole genes to exactly zero rather than scattering zeros across individual weights.

In [16]:
def fc1_only_penalty(model: HiddenOnlyMLP) -> torch.Tensor:
    return L1_LAMBDA_FC1 * model.fc1.weight.abs().sum()


def fc1_fc2_penalty(model: HiddenOnlyMLP) -> torch.Tensor:
    return L1_LAMBDA_FC1 * model.fc1.weight.abs().sum() + L1_LAMBDA_FC2 * model.fc2.weight.abs().sum()


def fc1_group_fc2_penalty(model: HiddenOnlyMLP) -> torch.Tensor:
    group_penalty = model.fc1.weight.norm(p=2, dim=0).sum()  # per-gene L2 norm across hidden units
    return L1_LAMBDA_FC1_GROUP * group_penalty + L1_LAMBDA_FC2 * model.fc2.weight.abs().sum()


def skip_penalty(model: SkipConnectionSparseMLP) -> torch.Tensor:
    return (L1_LAMBDA_DIRECT * model.direct.weight.abs().sum()
            + L1_LAMBDA_FC1 * model.fc1.weight.abs().sum())
            # + L1_LAMBDA_FC2 * model.fc2.weight.abs().sum())


VARIANTS = {
    'fc1_only':       {'model_cls': HiddenOnlyMLP,          'penalty_fn': fc1_only_penalty},
    'fc1_fc2':        {'model_cls': HiddenOnlyMLP,          'penalty_fn': fc1_fc2_penalty},
    'fc1_group_fc2':  {'model_cls': HiddenOnlyMLP,          'penalty_fn': fc1_group_fc2_penalty},
    'skip_connection':{'model_cls': SkipConnectionSparseMLP, 'penalty_fn': skip_penalty},
}

## Training loop -- generic over model class and penalty
Same GPU-train / CPU-validate device pattern as prior notebooks.

In [17]:
def fit_model(model: nn.Module,
              train_loader: DataLoader,
              val_loader: DataLoader,
              penalty_fn,
              num_epochs: int,
              patience: int,
              checkpoint_path,
              train_device: torch.device,
              verbose: bool = True):
    """Train with MSE + penalty_fn(model) on train_device; validate on CPU each epoch."""
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.to(train_device)
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(train_device), yb.to(train_device)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb) + penalty_fn(model)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
        train_loss = total_loss / len(train_loader.dataset)

        model.to('cpu')
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch}')
                break

        if verbose and epoch % 10 == 0:
            print(f'  Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}')

    model.load_state_dict(best_state)
    model.to('cpu')
    if checkpoint_path is not None:
        torch.save(model.state_dict(), checkpoint_path)
    return model, history

## Importance, cognate rank, and effective-gene helpers
Shared across variants. `compute_importance` dispatches on architecture: path-weight product for hidden-only models, the direct linear weight for the skip model.

In [18]:
def compute_importance(model: nn.Module, gene_names: list[str], protein_names: list[str]) -> pd.DataFrame:
    """Per-architecture importance matrix: |W2 @ W1| for hidden-only, |W_direct| for skip."""
    model.to('cpu')
    if isinstance(model, SkipConnectionSparseMLP):
        W = model.direct.weight.detach().numpy()
        return pd.DataFrame(np.abs(W), index=protein_names, columns=gene_names)
    W1 = model.fc1.weight.detach().numpy()
    W2 = model.fc2.weight.detach().numpy()
    return pd.DataFrame(np.abs(W2 @ W1), index=protein_names, columns=gene_names)


def cognate_gene_rank(importance_df: pd.DataFrame, gene_map: pd.DataFrame) -> pd.DataFrame:
    """Rank of each protein's cognate RNA gene by importance score (1 = top predictor)."""
    rows = []
    for _, row in gene_map.iterrows():
        gene, adt = row['gene'], row['adt_name']
        if adt not in importance_df.index or gene not in importance_df.columns:
            continue
        ranks = importance_df.loc[adt].rank(ascending=False)
        rows.append({'protein': adt, 'cognate_gene': gene, 'cognate_rank': int(ranks[gene])})
    return pd.DataFrame(rows)


def effective_genes_per_protein(model: nn.Module, protein_names: list[str], gene_names: list[str],
                                 thresh: float = SPARSITY_THRESH) -> pd.Series:
    """Median genes actually feeding each protein's prediction.

    Hidden-only: active hidden units (fc2) x active genes feeding them (fc1), union per protein.
    Skip: nonzero entries in that protein's direct-path weight row.
    """
    model.to('cpu')
    if isinstance(model, SkipConnectionSparseMLP):
        W = model.direct.weight.detach().numpy()
        counts = (np.abs(W) >= thresh).sum(axis=1)
        return pd.Series(counts, index=protein_names)

    W1 = model.fc1.weight.detach().numpy()
    W2 = model.fc2.weight.detach().numpy()
    counts = []
    for pi in range(len(protein_names)):
        active_units = np.where(np.abs(W2[pi, :]) >= thresh)[0]
        if len(active_units) == 0:
            counts.append(0)
            continue
        active_genes = (np.abs(W1[active_units, :]) >= thresh).any(axis=0)
        counts.append(int(active_genes.sum()))
    return pd.Series(counts, index=protein_names)


def evaluate_per_protein(model: nn.Module, X: np.ndarray, Y: np.ndarray, protein_names: list[str]) -> pd.DataFrame:
    """Per-protein Pearson r and R2 on CPU."""
    model.to('cpu')
    model.eval()
    with torch.no_grad():
        preds = model(torch.from_numpy(X)).numpy()
    rows = []
    for i, name in enumerate(protein_names):
        r, _ = pearsonr(Y[:, i], preds[:, i])
        r2 = r2_score(Y[:, i], preds[:, i])
        rows.append({'protein': name, 'pearson_r': r, 'r2': r2})
    return pd.DataFrame(rows)

## Train all four variants

In [19]:
trained_models = {}
trained_histories = {}

for name, cfg in VARIANTS.items():
    print(f'\n=== Training {name} ===')
    protein_dim = Y_protein_corr.shape[1]
    if cfg['model_cls'] is SkipConnectionSparseMLP:
        model = cfg['model_cls'](rna_dim=X_rna_corr.shape[1], hidden_dim=HIDDEN_DIM, protein_dim=protein_dim)
    else:
        model = cfg['model_cls'](rna_dim=X_rna_corr.shape[1], hidden_dim=HIDDEN_DIM, protein_dim=protein_dim)

    model, history = fit_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        penalty_fn=cfg['penalty_fn'], num_epochs=NUM_EPOCHS, patience=PATIENCE,
        checkpoint_path=MODELS_DIR / f'{name}.pt', train_device=DEVICE,
    )
    trained_models[name] = model
    trained_histories[name] = history


=== Training fc1_only ===
  Epoch 0: train_loss=0.3287, val_loss=0.2781
  Epoch 10: train_loss=0.2790, val_loss=0.2602
  Epoch 20: train_loss=0.2757, val_loss=0.2578
  Epoch 30: train_loss=0.2743, val_loss=0.2565
  Epoch 40: train_loss=0.2734, val_loss=0.2554
  Epoch 50: train_loss=0.2729, val_loss=0.2551
  Epoch 60: train_loss=0.2728, val_loss=0.2547
  Epoch 70: train_loss=0.2728, val_loss=0.2550
  Epoch 80: train_loss=0.2727, val_loss=0.2547
  Early stopping at epoch 83

=== Training fc1_fc2 ===
  Epoch 0: train_loss=0.3702, val_loss=0.2787
  Epoch 10: train_loss=0.2887, val_loss=0.2659
  Epoch 20: train_loss=0.2877, val_loss=0.2657
  Epoch 30: train_loss=0.2873, val_loss=0.2655
  Epoch 40: train_loss=0.2872, val_loss=0.2653
  Epoch 50: train_loss=0.2871, val_loss=0.2651
  Epoch 60: train_loss=0.2870, val_loss=0.2650
  Early stopping at epoch 70

=== Training fc1_group_fc2 ===
  Epoch 0: train_loss=0.3820, val_loss=0.2837
  Epoch 10: train_loss=0.3070, val_loss=0.2740
  Epoch 20: tr

## Comparison table -- accuracy vs. interpretability
Train and test Pearson r, cognate rank-1 count, and median effective genes per protein, side by side for all four variants.

In [20]:
comparison_rows = []

for name, model in trained_models.items():
    train_m = evaluate_per_protein(model, X_rna_corr[train_idx], Y_protein_corr[train_idx], matched_adt_names)
    test_m  = evaluate_per_protein(model, X_rna_corr[test_idx],  Y_protein_corr[test_idx],  matched_adt_names)

    importance_df = compute_importance(model, gene_union, matched_adt_names)
    cognate_ranks = cognate_gene_rank(importance_df, gene_map)
    eff_genes = effective_genes_per_protein(model, matched_adt_names, gene_union)

    comparison_rows.append({
        'variant': name,
        'median_train_r': train_m['pearson_r'].median(),
        'median_test_r': test_m['pearson_r'].median(),
        'r_gap': train_m['pearson_r'].median() - test_m['pearson_r'].median(),
        'cognate_rank1_count': int((cognate_ranks['cognate_rank'] == 1).sum()),
        'cognate_rank1_total': len(cognate_ranks),
        'median_effective_genes': eff_genes.median(),
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table

,variant,median_train_r,median_test_r,r_gap,cognate_rank1_count,cognate_rank1_total,median_effective_genes
0,fc1_only,0.446730,0.442159,0.004570,7,163,1699.0
1,fc1_fc2,0.402002,0.396751,0.005250,9,163,1355.0
2,fc1_group_fc2,0.366543,0.372877,-0.006334,3,163,1027.0
3,skip_connection,0.444329,0.435535,0.008795,42,163,386.0


## Pick variants for deep validation
Defaults to the best-accuracy variant and the best-interpretability variant (by cognate rank-1 count) from the table above. Override `VALIDATE_VARIANTS` manually if a different pair is more informative once you've looked at the table.

In [21]:
best_accuracy_variant = comparison_table.sort_values('median_test_r', ascending=False).iloc[0]['variant']
best_interpretability_variant = comparison_table.sort_values('cognate_rank1_count', ascending=False).iloc[0]['variant']

VALIDATE_VARIANTS = sorted(set([best_accuracy_variant, best_interpretability_variant]))
print(f'Validating: {VALIDATE_VARIANTS}')

Validating: ['fc1_only', 'skip_connection']


## Bootstrap stability -- aggregate top-K AND rank-1-specific
Two scores per gene: whether it recurs in the top-`TOP_K_GENES` across bootstraps (as before), and separately, whether each protein's single #1 gene stays #1. The rank-1-specific check is the more decision-relevant number -- the aggregate metric mixes a very stable signal (rank 1) with noisier ones (ranks 2-20).

In [22]:
def train_on_bootstrap(variant_name: str, seed: int) -> nn.Module:
    """Train a given variant on a bootstrap resample of the training cells (val/test fixed)."""
    cfg = VARIANTS[variant_name]
    rng = np.random.RandomState(seed)
    boot_idx = rng.choice(train_idx, size=len(train_idx), replace=True)
    boot_loader = make_loader(X_rna_corr, Y_protein_corr, boot_idx, BATCH_SIZE, shuffle=True)

    model = cfg['model_cls'](rna_dim=X_rna_corr.shape[1], hidden_dim=HIDDEN_DIM, protein_dim=Y_protein_corr.shape[1])
    model, _ = fit_model(
        model=model, train_loader=boot_loader, val_loader=val_loader,
        penalty_fn=cfg['penalty_fn'], num_epochs=AUX_NUM_EPOCHS, patience=AUX_PATIENCE,
        checkpoint_path=None, train_device=DEVICE, verbose=False,
    )
    return model


bootstrap_results = {}  # variant -> {'top_k_sets': {protein: [set,...]}, 'top1_genes': {protein: [gene,...]}}

for variant_name in VALIDATE_VARIANTS:
    print(f'\n--- Bootstrapping {variant_name} ---')
    top_k_sets = {protein: [] for protein in matched_adt_names}
    top1_genes = {protein: [] for protein in matched_adt_names}

    for b in range(N_BOOTSTRAPS):
        print(f'  Bootstrap {b + 1}/{N_BOOTSTRAPS}...')
        boot_model = train_on_bootstrap(variant_name, seed=2000 + b)
        boot_importance = compute_importance(boot_model, gene_union, matched_adt_names)
        for protein in matched_adt_names:
            ranked = boot_importance.loc[protein].sort_values(ascending=False)
            top_k_sets[protein].append(set(ranked.head(TOP_K_GENES).index))
            top1_genes[protein].append(ranked.index[0])

    bootstrap_results[variant_name] = {'top_k_sets': top_k_sets, 'top1_genes': top1_genes}

print('\nBootstrap runs complete.')


--- Bootstrapping fc1_only ---
  Bootstrap 1/5...
  Bootstrap 2/5...
  Bootstrap 3/5...
  Bootstrap 4/5...
  Bootstrap 5/5...

--- Bootstrapping skip_connection ---
  Bootstrap 1/5...
  Bootstrap 2/5...
  Bootstrap 3/5...
  Bootstrap 4/5...
  Bootstrap 5/5...

Bootstrap runs complete.


In [23]:
def compute_stability_scores(variant_name: str, model: nn.Module) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Aggregate top-K bootstrap frequency, and rank-1-specific match frequency."""
    importance_df = compute_importance(model, gene_union, matched_adt_names)
    top_k_sets = bootstrap_results[variant_name]['top_k_sets']
    top1_genes_boot = bootstrap_results[variant_name]['top1_genes']

    agg_rows = []
    rank1_rows = []
    for protein in matched_adt_names:
        ranked = importance_df.loc[protein].sort_values(ascending=False)
        ref_top1 = ranked.index[0]

        for gene, score in ranked.head(TOP_K_GENES).items():
            freq = sum(gene in s for s in top_k_sets[protein]) / len(top_k_sets[protein])
            agg_rows.append({'protein': protein, 'gene': gene, 'importance': score, 'bootstrap_frequency': freq})

        rank1_match_freq = sum(g == ref_top1 for g in top1_genes_boot[protein]) / len(top1_genes_boot[protein])
        rank1_rows.append({'protein': protein, 'reference_top1_gene': ref_top1, 'rank1_match_frequency': rank1_match_freq})

    return pd.DataFrame(agg_rows), pd.DataFrame(rank1_rows)


stability_by_variant = {}
rank1_stability_by_variant = {}

for variant_name in VALIDATE_VARIANTS:
    agg_df, rank1_df = compute_stability_scores(variant_name, trained_models[variant_name])
    stability_by_variant[variant_name] = agg_df
    rank1_stability_by_variant[variant_name] = rank1_df

    print(f'\n{variant_name}:')
    print(f'  Aggregate top-{TOP_K_GENES} -- median bootstrap frequency: {agg_df["bootstrap_frequency"].median():.2f}, '
          f'fraction >=80%: {(agg_df["bootstrap_frequency"] >= 0.8).mean():.1%}')
    print(f'  Rank-1 specific -- median match frequency: {rank1_df["rank1_match_frequency"].median():.2f}, '
          f'fraction >=80%: {(rank1_df["rank1_match_frequency"] >= 0.8).mean():.1%}')


fc1_only:
  Aggregate top-20 -- median bootstrap frequency: 0.80, fraction >=80%: 59.3%
  Rank-1 specific -- median match frequency: 1.00, fraction >=80%: 77.9%

skip_connection:
  Aggregate top-20 -- median bootstrap frequency: 0.00, fraction >=80%: 5.4%
  Rank-1 specific -- median match frequency: 0.00, fraction >=80%: 28.8%


## Permutation null baseline
Same as prior notebooks -- shuffle protein rows, retrain, use resulting importance as an empirical significance threshold. Run per validated variant.

In [24]:
def train_on_permuted_protein(variant_name: str, seed: int) -> nn.Module:
    """Train a given variant with protein rows shuffled (real RNA-protein pairing destroyed)."""
    cfg = VARIANTS[variant_name]
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n_cells)
    Y_perm = Y_protein_corr[perm]

    perm_train_loader = make_loader(X_rna_corr, Y_perm, train_idx, BATCH_SIZE, shuffle=True)
    perm_val_loader   = make_loader(X_rna_corr, Y_perm, val_idx,   BATCH_SIZE, shuffle=False)

    model = cfg['model_cls'](rna_dim=X_rna_corr.shape[1], hidden_dim=HIDDEN_DIM, protein_dim=Y_protein_corr.shape[1])
    model, _ = fit_model(
        model=model, train_loader=perm_train_loader, val_loader=perm_val_loader,
        penalty_fn=cfg['penalty_fn'], num_epochs=AUX_NUM_EPOCHS, patience=AUX_PATIENCE,
        checkpoint_path=None, train_device=DEVICE, verbose=False,
    )
    return model


null_thresholds_by_variant = {}

for variant_name in VALIDATE_VARIANTS:
    print(f'\n--- Null permutations for {variant_name} ---')
    null_importances = []
    for n in range(N_NULL_PERMS):
        print(f'  Null permutation {n + 1}/{N_NULL_PERMS}...')
        null_model = train_on_permuted_protein(variant_name, seed=3000 + n)
        null_importances.append(compute_importance(null_model, gene_union, matched_adt_names))

    thresholds = {}
    for protein in matched_adt_names:
        pooled_null = np.concatenate([df.loc[protein].values for df in null_importances])
        thresholds[protein] = np.percentile(pooled_null, 95.0)
    null_thresholds_by_variant[variant_name] = pd.Series(thresholds)

print('\nNull runs complete.')


--- Null permutations for fc1_only ---
  Null permutation 1/5...
  Null permutation 2/5...
  Null permutation 3/5...
  Null permutation 4/5...
  Null permutation 5/5...

--- Null permutations for skip_connection ---
  Null permutation 1/5...
  Null permutation 2/5...
  Null permutation 3/5...
  Null permutation 4/5...
  Null permutation 5/5...

Null runs complete.


In [25]:
validated_by_variant = {}

for variant_name in VALIDATE_VARIANTS:
    agg_df = stability_by_variant[variant_name]
    thresholds = null_thresholds_by_variant[variant_name]

    validated = agg_df.merge(thresholds.rename('null_threshold'), left_on='protein', right_index=True)
    validated['above_null'] = validated['importance'] > validated['null_threshold']
    validated['validated'] = validated['above_null'] & (validated['bootstrap_frequency'] >= 0.8)
    validated_by_variant[variant_name] = validated

    rank1_df = rank1_stability_by_variant[variant_name]
    print(f'\n{variant_name}:')
    print(f'  Median null threshold: {thresholds.median():.6f}')
    print(f'  Validated pairs (aggregate top-K criterion): {validated["validated"].sum()} / {len(validated)}')
    print(f'  Proteins with a stable (>=80%) rank-1 gene: {(rank1_df["rank1_match_frequency"] >= 0.8).sum()} / {len(rank1_df)}')


fc1_only:
  Median null threshold: 0.000045
  Validated pairs (aggregate top-K criterion): 1934 / 3260
  Proteins with a stable (>=80%) rank-1 gene: 127 / 163

skip_connection:
  Median null threshold: 0.002379
  Validated pairs (aggregate top-K criterion): 176 / 3260
  Proteins with a stable (>=80%) rank-1 gene: 47 / 163


## Save results

In [26]:
comparison_table.to_csv(RESULTS_DIR / 'architecture_comparison.csv', index=False)

for variant_name in VALIDATE_VARIANTS:
    safe_name = variant_name.replace(' ', '_')
    stability_by_variant[variant_name].to_csv(RESULTS_DIR / f'{safe_name}_bootstrap_stability.csv', index=False)
    rank1_stability_by_variant[variant_name].to_csv(RESULTS_DIR / f'{safe_name}_rank1_stability.csv', index=False)
    validated_by_variant[variant_name].to_csv(RESULTS_DIR / f'{safe_name}_validated.csv', index=False)

print(f'Saved to {RESULTS_DIR}')
print('  architecture_comparison.csv              -- all 4 variants, accuracy + interpretability side by side')
for variant_name in VALIDATE_VARIANTS:
    print(f'  {variant_name}_bootstrap_stability.csv  -- aggregate top-{TOP_K_GENES} stability')
    print(f'  {variant_name}_rank1_stability.csv      -- rank-1-specific stability')
    print(f'  {variant_name}_validated.csv            -- bootstrap + null validated pairs')

Saved to /content/drive/MyDrive/covid_citeseq_project/results/sparse_mlp_comparison
  architecture_comparison.csv              -- all 4 variants, accuracy + interpretability side by side
  fc1_only_bootstrap_stability.csv  -- aggregate top-20 stability
  fc1_only_rank1_stability.csv      -- rank-1-specific stability
  fc1_only_validated.csv            -- bootstrap + null validated pairs
  skip_connection_bootstrap_stability.csv  -- aggregate top-20 stability
  skip_connection_rank1_stability.csv      -- rank-1-specific stability
  skip_connection_validated.csv            -- bootstrap + null validated pairs
